# Qwen2.5-Coder-14B-Instruct on Google Colab (Free GPU)
This notebook loads Qwen2.5-Coder-14B-Instruct with vLLM and exposes it as an OpenAI-compatible API endpoint.

## 1. Setup Environment

In [ ]:
!pip install -q torch transformers accelerate sentencepiece vllm==0.6.3.post1 fastapi uvicorn python-multipart

⚠️ **CRITICAL INSTRUCTION: RESTART SESSION** ⚠️
After running the `pip install` cell above, you **MUST** restart the Colab session. 
Go to **Runtime > Restart session** (or Restart runtime), and then run the remaining cells starting from Step 2.

## 2. Check GPU

In [ ]:
import torch
print(f'GPU available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

## 3. Load Model with vLLM
We use the AWQ quantized version to fit within the free T4 GPU's 16GB VRAM.

In [ ]:
from vllm import LLM, SamplingParams

model_name = "Qwen/Qwen2.5-Coder-14B-Instruct-AWQ"
llm = LLM(
    model=model_name,
    trust_remote_code=True,
    tensor_parallel_size=1,
    max_model_len=4096,
    enforce_eager=True,
    gpu_memory_utilization=0.85
)


## 4. Expose as FastAPI Endpoint

In [ ]:
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import uvicorn
import json

app = FastAPI()

@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    body = await request.json()
    prompts = [body["messages"][-1]["content"]]  # Extract latest user message
    sampling_params = SamplingParams(temperature=0.7, max_tokens=1024)
    outputs = llm.generate(prompts, sampling_params)
    response = {
        "choices": [{
            "message": {"role": "assistant", "content": outputs[0].outputs[0].text},
            "finish_reason": "stop"
        }]
    }
    return JSONResponse(content=response)


## 5. Expose via Cloudflare Tunnel (No Auth Required)

In [ ]:
!wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

import subprocess
import threading
import time

# Start FastAPI in a background thread
def run_fastapi():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_fastapi, daemon=True).start()

time.sleep(2)

# Start cloudflared
process = subprocess.Popen(['./cloudflared-linux-amd64', 'tunnel', '--url', 'http://127.0.0.1:8000'],
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print("Looking for Cloudflare tunnel URL...")
while True:
    line = process.stdout.readline()
    if not line:
        break
    if "trycloudflare.com" in line:
        url = line.split("trycloudflare.com")[0].split("https://")[-1] + "trycloudflare.com"
        print(f"\n>>> Public endpoint: https://{url}/v1 <<<\n")
        break

while True:
    time.sleep(1)
